In [1]:
import re
import joblib
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

In [2]:
# ===== Step 1: Labeled training data (term -> category) =====
data = [
    # SKILL
    ("Python", "SKILL"), ("SQL", "SKILL"), ("Java", "SKILL"), ("Scala", "SKILL"),
    ("R", "SKILL"), ("Machine Learning", "SKILL"), ("Deep Learning", "SKILL"),
    ("Natural Language Processing", "SKILL"), ("Statistics", "SKILL"),
    ("Data Analysis", "SKILL"), ("Data Visualization", "SKILL"), ("Big Data", "SKILL"),
    # CLOUD
    ("AWS", "CLOUD"), ("Azure", "CLOUD"), ("Google Cloud", "CLOUD"), ("GCP", "CLOUD"),
    ("Microsoft Azure", "CLOUD"), ("AWS Lambda", "CLOUD"), ("AWS EC2", "CLOUD"),
    # DATABASE
    ("PostgreSQL", "DATABASE"), ("MongoDB", "DATABASE"), ("MySQL", "DATABASE"),
    ("SQL Server", "DATABASE"), ("Redis", "DATABASE"), ("Snowflake", "DATABASE"),
    ("BigQuery", "DATABASE"), ("Elasticsearch", "DATABASE"),
    # BI_TOOL
    ("Power BI", "BI_TOOL"), ("Tableau", "BI_TOOL"), ("Looker", "BI_TOOL"),
    ("Excel", "BI_TOOL"), ("QlikView", "BI_TOOL"),
    # ML_FRAMEWORK
    ("TensorFlow", "ML_FRAMEWORK"), ("PyTorch", "ML_FRAMEWORK"), ("Keras", "ML_FRAMEWORK"),
    ("Scikit-learn", "ML_FRAMEWORK"), ("XGBoost", "ML_FRAMEWORK"),
    # DEVOPS_TOOL
    ("Docker", "DEVOPS_TOOL"), ("Kubernetes", "DEVOPS_TOOL"), ("Jenkins", "DEVOPS_TOOL"),
    ("Git", "DEVOPS_TOOL"), ("Terraform", "DEVOPS_TOOL"),
    # BIG_DATA_TOOL
    ("Spark", "BIG_DATA_TOOL"), ("Hadoop", "BIG_DATA_TOOL"), ("Kafka", "BIG_DATA_TOOL"),
    ("Airflow", "BIG_DATA_TOOL"),
    # NOT_SKILL - critical: without this, the model is FORCED to pick a skill
    # category for every word, including generic non-skill words like "experience".
    ("experience", "NOT_SKILL"), ("team", "NOT_SKILL"), ("work", "NOT_SKILL"),
    ("business", "NOT_SKILL"), ("years", "NOT_SKILL"), ("degree", "NOT_SKILL"),
    ("required", "NOT_SKILL"), ("support", "NOT_SKILL"), ("company", "NOT_SKILL"),
    ("communication", "NOT_SKILL"), ("leadership", "NOT_SKILL"), ("management", "NOT_SKILL"),
]
df = pd.DataFrame(data, columns=["term", "category"])

# ===== Step 2: Preprocessing =====
def preprocess(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

df["clean_term"] = df["term"].apply(preprocess)

# ===== Step 3: Train/test split =====
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_term"], df["category"],
    test_size=0.25, random_state=42, stratify=df["category"]
)

# ===== Step 4: Pipeline — Text -> TF-IDF -> Logistic Regression -> Category =====
# char_wb n-grams work much better than word-level TF-IDF for short skill terms
# (word-level TF-IDF treats "Power BI" as two nearly-unique tokens with no signal).
skill_classifier = Pipeline([
    ("tfidf", TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), min_df=1)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
])
skill_classifier.fit(X_train, y_train)

# ===== Step 5: Evaluate =====
y_pred = skill_classifier.predict(X_test)
print("Test accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred, zero_division=0))

# ===== Step 6: Save the deliverable =====
joblib.dump(skill_classifier, "skill_classifier.pkl")
print("Saved skill_classifier.pkl")

# ===== Step 7: Apply to real candidate terms from your job description corpus =====
# skill_classifier = joblib.load("skill_classifier.pkl")  # to reload later
new_terms = ["Java", "Snowflake", "Kubernetes", "experience", "Ansible", "Power BI"]
preds = skill_classifier.predict([preprocess(t) for t in new_terms])
for t, p in zip(new_terms, preds):
    print(f"{t!r:20s} -> predicted category: {p}")

Test accuracy: 0.2
               precision    recall  f1-score   support

BIG_DATA_TOOL       0.00      0.00      0.00         1
      BI_TOOL       0.00      0.00      0.00         2
        CLOUD       1.00      0.50      0.67         2
     DATABASE       0.00      0.00      0.00         2
  DEVOPS_TOOL       0.00      0.00      0.00         1
 ML_FRAMEWORK       0.00      0.00      0.00         1
    NOT_SKILL       0.25      0.33      0.29         3
        SKILL       0.17      0.33      0.22         3

     accuracy                           0.20        15
    macro avg       0.18      0.15      0.15        15
 weighted avg       0.22      0.20      0.19        15

Saved skill_classifier.pkl
'Java'               -> predicted category: SKILL
'Snowflake'          -> predicted category: DATABASE
'Kubernetes'         -> predicted category: DEVOPS_TOOL
'experience'         -> predicted category: NOT_SKILL
'Ansible'            -> predicted category: SKILL
'Power BI'           -> pred